# ow-0009 (April 2019) -- AIS-2 event-window track preparation/validation

**Same unmodified `iste.ais_data` module as the June event's AIS-2 -- only the
event window changes.** Prepares/validates the real historical AIS CSV against
the April 25-28, 2019 event window and the same project AOI. Does NOT rank
vessels or compute an AIS evidence score.

In [1]:
import os, sys, time
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iste.ais_data import (
    load_ais_csv, compute_data_quality_report, filter_to_event_window, filter_to_aoi,
    build_vessel_tracks, compute_track_temporal_gaps, compute_track_positional_jumps,
    tracks_to_dataframe, DEFAULT_MAX_PLAUSIBLE_SPEED_KNOTS,
)

RESULT_DIR = os.path.join(REPO_ROOT, "result_ow0009")
os.makedirs(RESULT_DIR, exist_ok=True)

T_OBS = pd.Timestamp("2019-04-28T15:38:41Z")
T72 = pd.Timestamp("2019-04-25T15:38:41Z")
AOI_NORTH, AOI_WEST, AOI_SOUTH, AOI_EAST = 36.372, 27.121, 30.1875, 36.088  # same project AOI, unchanged
print(f"event window: T72={T72} .. T0={T_OBS}")
print(f"project AOI: N={AOI_NORTH} W={AOI_WEST} S={AOI_SOUTH} E={AOI_EAST}")

event window: T72=2019-04-25 15:38:41+00:00 .. T0=2019-04-28 15:38:41+00:00
project AOI: N=36.372 W=27.121 S=30.1875 E=36.088


## 1. Load AIS CSV (full dataset already validated in AIS-2 for June; reused unchanged)

In [2]:
AIS_CSV_PATH = os.path.join(REPO_ROOT, "east_med_massive_random_ais.csv")
t0 = time.time()
ais_df = load_ais_csv(AIS_CSV_PATH)
t1 = time.time()
print(f"loaded {len(ais_df)} rows in {t1-t0:.1f}s")
report = compute_data_quality_report(ais_df)
print(f"timestamp range: {report.timestamp_min} .. {report.timestamp_max}")
print(f"n_duplicate_mmsi_timestamp: {report.n_duplicate_mmsi_timestamp}, n_invalid_lat/lon: {report.n_invalid_lat}/{report.n_invalid_lon}")

loaded 124887 rows in 0.3s


timestamp range: 2019-01-10 17:05:00+00:00 .. 2019-12-17 07:11:39+00:00
n_duplicate_mmsi_timestamp: 0, n_invalid_lat/lon: 0/0


## 2. Extract event-window + AOI records

In [3]:
windowed = filter_to_event_window(ais_df, T72, T_OBS)
print(f"rows in event window [T72, T0]: {len(windowed)} / {len(ais_df)}")
print(f"distinct MMSI in event window (before AOI filter): {windowed['mmsi'].nunique()}")

event_df = filter_to_aoi(windowed, north=AOI_NORTH, south=AOI_SOUTH, east=AOI_EAST, west=AOI_WEST)
print(f"rows in event window AND project AOI: {len(event_df)} / {len(windowed)}")
print(f"distinct MMSI in event window AND AOI: {event_df['mmsi'].nunique()}")

rows in event window [T72, T0]: 8088 / 124887
distinct MMSI in event window (before AOI filter): 218
rows in event window AND project AOI: 8017 / 8088
distinct MMSI in event window AND AOI: 217


## 3. Build one chronological track per MMSI

In [4]:
event_tracks = build_vessel_tracks(event_df)
print(f"built {len(event_tracks)} vessel tracks from the event-window+AOI subset")
n_obs_total = sum(t.n_observations for t in event_tracks)
assert n_obs_total == len(event_df)
obs_counts = np.array([t.n_observations for t in event_tracks])
n_single_obs = int(np.sum(obs_counts == 1))
print(f"observations per track: min={obs_counts.min()}, median={np.median(obs_counts):.1f}, mean={obs_counts.mean():.2f}, max={obs_counts.max()}")
print(f"tracks with only 1 observation: {n_single_obs}")

built 217 vessel tracks from the event-window+AOI subset
observations per track: min=1, median=37.0, mean=36.94, max=73
tracks with only 1 observation: 4


## 4. Temporal-gap + positional-jump validation (flag, never delete)

In [5]:
GAP_THRESHOLD_HOURS = 6.0
all_gaps = [compute_track_temporal_gaps(t, gap_threshold_hours=GAP_THRESHOLD_HOURS) for t in event_tracks]
n_tracks_with_gap = sum(1 for g in all_gaps if g.n_gaps_over_threshold > 0)
print(f"tracks with >=1 gap exceeding {GAP_THRESHOLD_HOURS}h: {n_tracks_with_gap} / {len(event_tracks)}")

all_jumps = []
for t in event_tracks:
    all_jumps.extend(compute_track_positional_jumps(t, max_plausible_speed_knots=DEFAULT_MAX_PLAUSIBLE_SPEED_KNOTS))
implausible = [j for j in all_jumps if j.is_implausible]
print(f"flagged implausible jumps (>{DEFAULT_MAX_PLAUSIBLE_SPEED_KNOTS} knots): {len(implausible)} / {len(all_jumps)} (none deleted)")

if implausible:
    jump_rows = [dict(mmsi=j.mmsi, t_prev=str(j.t_prev), t_curr=str(j.t_curr),
                       distance_m=round(j.distance_m, 1), interval_hours=round(j.interval_hours, 4),
                       implied_speed_knots=round(j.implied_speed_knots, 2), reported_sog_knots=j.reported_sog_knots)
                  for j in implausible]
    jump_df = pd.DataFrame(jump_rows).sort_values("implied_speed_knots", ascending=False)
else:
    jump_df = pd.DataFrame(columns=["mmsi","t_prev","t_curr","distance_m","interval_hours","implied_speed_knots","reported_sog_knots"])
jump_df.to_csv(os.path.join(RESULT_DIR, "AIS_positional_jump_flags.csv"), index=False)

tracks with >=1 gap exceeding 6.0h: 0 / 217


flagged implausible jumps (>60.0 knots): 0 / 7800 (none deleted)


## 5. Save the event-window+AOI vessel-track dataset

In [6]:
track_df = tracks_to_dataframe(event_tracks)
assert len(track_df) == len(event_df)
track_csv_path = os.path.join(RESULT_DIR, "AIS_vessel_tracks_event_window.csv")
track_df.to_csv(track_csv_path, index=False)
print(f"saved {track_csv_path} ({len(track_df)} rows, {track_df['mmsi'].nunique()} vessels)")

saved C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result_ow0009\AIS_vessel_tracks_event_window.csv (8017 rows, 217 vessels)


## 6. Event-window summary + validation

In [7]:
n_vessels_in_window_before_aoi = int(windowed["mmsi"].nunique())
n_usable_tracks = int(np.sum(obs_counts >= 2))
summary_lines = [
    "ow-0009 (April 2019) -- AIS event-window summary",
    f"event window: T72={T72} .. T0={T_OBS}",
    f"project AOI: N={AOI_NORTH} W={AOI_WEST} S={AOI_SOUTH} E={AOI_EAST}",
    "",
    f"vessels with >=1 AIS position inside the event window (before AOI filter): {n_vessels_in_window_before_aoi}",
    f"vessels entering the AOI during the event window: {len(event_tracks)}",
    f"AIS observations in event window + AOI: {len(event_df)}",
    f"usable tracks (>=2 observations): {n_usable_tracks}",
    f"single-observation tracks: {n_single_obs}",
    f"tracks with >=1 gap exceeding {GAP_THRESHOLD_HOURS}h: {n_tracks_with_gap}",
    f"flagged implausible positional jumps: {len(implausible)}",
]
print("\n".join(summary_lines))
with open(os.path.join(RESULT_DIR, "AIS_event_window_summary.txt"), "w", encoding="utf-8") as f:
    f.write("\n".join(summary_lines))

checks = []
checks.append(("all event-window+AOI rows accounted for in tracks", sum(t.n_observations for t in event_tracks) == len(event_df)))
checks.append(("output files written under result_ow0009/", os.path.exists(track_csv_path)))
for label, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks)

ow-0009 (April 2019) -- AIS event-window summary
event window: T72=2019-04-25 15:38:41+00:00 .. T0=2019-04-28 15:38:41+00:00
project AOI: N=36.372 W=27.121 S=30.1875 E=36.088

vessels with >=1 AIS position inside the event window (before AOI filter): 218
vessels entering the AOI during the event window: 217
AIS observations in event window + AOI: 8017
usable tracks (>=2 observations): 213
single-observation tracks: 4
tracks with >=1 gap exceeding 6.0h: 0
flagged implausible positional jumps: 0
[OK] all event-window+AOI rows accounted for in tracks
[OK] output files written under result_ow0009/
